# vllm-lens Benchmark: Overhead of Activation Capture & Steering

Measures the performance overhead of vllm-lens features compared to plain vLLM generation.

A single `AsyncLLMEngine` is used throughout — the vllm-lens plugin is always loaded
(via entry point) but only activates when `extra_args` contains
`output_residual_stream` or `apply_steering_vectors`. The baseline is generation
with no `extra_args`, i.e. effectively plain vLLM.

| # | Benchmark | Description |
|---|-----------|-------------|
| 1 | Baseline | Pure generation, no vllm-lens features |
| 2 | One layer | Capture residual stream from 1 layer |
| 3 | All layers | Capture residual stream from all 80 layers |
| 4 | Random layer | Each request captures a random layer (0–10) |
| 5 | Steering | Apply a steering vector at layer 5 |
| 6 | LoRA alt | Alternate LoRA adapter on/off per request |

In [1]:
import random
import time
from collections.abc import Callable
from dataclasses import dataclass

import torch
from datasets import load_dataset
from tqdm.asyncio import tqdm
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams
from vllm.lora.request import LoRARequest

MODEL = "meta-llama/Llama-3.3-70B-Instruct"
GPU_MEM = 0.85
NUM_PROMPTS = 50
MAX_TOKENS = 1024
TEMPERATURE = 0.0
SEED = 42
TP_SIZE = 4

# Llama 3.3 70B: 80 decoder layers (0-79), 8192 hidden dim
NUM_LAYERS = 80
HIDDEN_DIM = 8192

LORA_PATH = "adamkarvonen/checkpoints_act_cls_latentqa_pretrain_mix_adding_Llama-3_3-70B-Instruct"

## Load Dataset

In [2]:
ds = load_dataset("tatsu-lab/alpaca", split="train")
prompts = [ds[i]["instruction"] for i in range(NUM_PROMPTS)]
print(f"Loaded {len(prompts)} prompts")
print(f"Example: {prompts[0][:100]}...")

Loaded 50 prompts
Example: Give three tips for staying healthy....


## Initialize Engine

In [3]:
engine_args = AsyncEngineArgs(
    model=MODEL,
    gpu_memory_utilization=GPU_MEM,
    tensor_parallel_size=TP_SIZE,
    enable_lora=True,
    max_lora_rank=64,
    max_model_len=MAX_TOKENS,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)

INFO 03-14 10:30:41 [model.py:531] Resolved architecture: LlamaForCausalLM
INFO 03-14 10:30:41 [model.py:1554] Using max model len 1024
INFO 03-14 10:30:41 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 03-14 10:30:41 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 03-14 10:30:41 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-14 10:30:41 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-14 10:30:46 [vllm.py:957] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=35711) INFO 03-14 10:30:48 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='meta-llama/Llama-3.3-70B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.3-70B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, rev

(EngineCore_DP0 pid=35711) (EngineCore_DP0 pid=35711) (Worker pid=35729) (Worker pid=35735) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore_DP0 pid=35711) (EngineCore_DP0 pid=35711) (Worker pid=35729) (Worker pid=35735) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(EngineCore_DP0 pi

(EngineCore_DP0 pid=35711) (Worker pid=35729) INFO 03-14 10:30:56 [pynccl.py:111] vLLM is using nccl==2.27.5


(EngineCore_DP0 pid=35711) (Worker pid=35733) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore_DP0 pid=35711) (Worker pid=35733) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore_DP0 pid=35711) (EngineCore_DP0 pid=35711) (EngineCore_DP0 pid=35711) (Worker pid=35729) (Worker pid=35735) (Worker pid=35733) WARNING 03-14 10:30:59 [symm_mem.py:107] SymmMemCommunicator: symmetric memory multicast operations are not supported.
WARNING 03-14 10:30:59 [symm_mem.py:107] SymmMemCommunicator: symmetric memory multicast operations are not supported.
WARNING 03-14 10:30:59 [symm_mem.py:107] SymmMemCommunicator: symmetric memory multicast operations are not supported.
(EngineCore_DP0 pid=35711) (Worker pid=35731) WARNING 03-14 10:30:59 [symm_mem.py:107] SymmMemCommunicator: symmetric memory multicast operations are not supported.
(EngineCore_DP0 pid=35711) (EngineCore_DP0 pid=35711) (EngineCore_DP0 pid=35711) (EngineCore_DP0 pid=35711) (Worker pid=35731) (Worker pid=35735) (Worker pid=35733) (Worker pid=35729) INFO 03-14 10:30:59 [parallel_state.py:1715] rank 1 in world size 4 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 1, EP rank N/A, EPLB rank N/A
I

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

(EngineCore_DP0 pid=35711) (Worker pid=35733) (Worker_TP2 pid=35733) WARNING 03-14 10:45:00 [utils.py:268] Using default LoRA kernel configs
(EngineCore_DP0 pid=35711) (Worker pid=35731) (Worker_TP1 pid=35731) WARNING 03-14 10:45:00 [utils.py:268] Using default LoRA kernel configs
(EngineCore_DP0 pid=35711) (Worker pid=35735) (Worker_TP3 pid=35735) WARNING 03-14 10:45:00 [utils.py:268] Using default LoRA kernel configs
(EngineCore_DP0 pid=35711) (Worker pid=35729) (Worker_TP0 pid=35729) WARNING 03-14 10:45:00 [utils.py:268] Using default LoRA kernel configs


## Benchmark Helpers

`run_benchmark` fires all requests concurrently, tracks progress with
`tqdm.gather`, and returns a `BenchmarkResult`.

The `make_params` factory lets each benchmark define per-request
`SamplingParams` (e.g. different layers per request) without duplicating
the benchmark loop.

In [4]:
@dataclass
class BenchmarkResult:
    """Results from a single benchmark run."""

    name: str
    wall_time: float
    num_requests: int
    total_output_tokens: int
    requests_per_second: float
    tokens_per_second: float


async def _generate_one(
    engine: AsyncLLMEngine,
    prompt: str,
    params: SamplingParams,
    request_id: str,
    lora_request: LoRARequest | None = None,
) -> int:
    """Run a single request, return number of output tokens."""
    final = None
    async for output in engine.generate(
        prompt, params, request_id, lora_request=lora_request
    ):
        final = output
    return len(final.outputs[0].token_ids)


async def run_benchmark(
    engine: AsyncLLMEngine,
    prompts: list[str],
    make_params: Callable[[int], SamplingParams],
    name: str,
    make_lora: Callable[[int], LoRARequest | None] | None = None,
) -> BenchmarkResult:
    """Fire all requests concurrently with tqdm progress and return timing results."""

    async def _task(idx: int) -> int:
        lora = make_lora(idx) if make_lora else None
        return await _generate_one(
            engine, prompts[idx], make_params(idx), f"{name}-{idx}", lora
        )

    start = time.perf_counter()
    token_counts = await tqdm.gather(
        *[_task(i) for i in range(len(prompts))],
        desc=name,
    )
    elapsed = time.perf_counter() - start

    total_tokens = sum(token_counts)
    return BenchmarkResult(
        name=name,
        wall_time=elapsed,
        num_requests=len(prompts),
        total_output_tokens=total_tokens,
        requests_per_second=len(prompts) / elapsed,
        tokens_per_second=total_tokens / elapsed,
    )


def compare(baseline: BenchmarkResult, lens: BenchmarkResult) -> None:
    """Print a comparison table with overhead percentages."""
    oh_time = (lens.wall_time - baseline.wall_time) / baseline.wall_time * 100
    oh_tps = (
        (baseline.tokens_per_second - lens.tokens_per_second)
        / baseline.tokens_per_second
        * 100
    )

    print(f"\n{'Metric':<25} {'Baseline':>12} {'Lens':>12} {'Overhead':>10}")
    print("-" * 61)
    print(
        f"{'Wall time (s)':<25} {baseline.wall_time:>12.2f} {lens.wall_time:>12.2f} {oh_time:>+9.1f}%"
    )
    print(
        f"{'Requests/s':<25} {baseline.requests_per_second:>12.1f} {lens.requests_per_second:>12.1f}"
    )
    print(
        f"{'Tokens/s':<25} {baseline.tokens_per_second:>12.1f} {lens.tokens_per_second:>12.1f} {oh_tps:>+9.1f}%"
    )
    print(
        f"{'Total output tokens':<25} {baseline.total_output_tokens:>12} {lens.total_output_tokens:>12}"
    )

## Warmup

In [5]:
warmup_params = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
_ = await run_benchmark(engine, prompts[:10], lambda i: warmup_params, "warmup")

warmup:   0%|          | 0/10 [00:00<?, ?it/s]

WARNING 03-14 10:31:27 [input_processor.py:254] Passing raw prompts to InputProcessor is deprecated and will be removed in v0.18. You should instead pass the outputs of Renderer.render_cmpl() or Renderer.render_chat().


warmup: 100%|██████████| 10/10 [01:30<00:00,  9.05s/it]


## Benchmark 1 — Pure Generation (Baseline)

No activation capture, no steering. Establishes baseline throughput.

In [6]:
baseline_params = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
baseline = await run_benchmark(engine, prompts, lambda i: baseline_params, "baseline")
print(
    f"\n{baseline.wall_time:.2f}s | {baseline.tokens_per_second:.1f} tok/s | {baseline.requests_per_second:.1f} req/s"
)

baseline: 100%|██████████| 50/50 [01:22<00:00,  1.64s/it]


82.15s | 413.6 tok/s | 0.6 req/s


## Benchmark 2 — One-Layer Activation Capture

Capture residual-stream activations from a single layer (layer 5).

In [7]:
one_layer_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    extra_args={"output_residual_stream": [5]},
)
one_layer = await run_benchmark(
    engine, prompts, lambda i: one_layer_params, "one-layer"
)
compare(baseline, one_layer)

one-layer: 100%|██████████| 50/50 [01:32<00:00,  1.84s/it]


Metric                        Baseline         Lens   Overhead
-------------------------------------------------------------
Wall time (s)                    82.15        92.02     +12.0%
Requests/s                         0.6          0.5
Tokens/s                         413.6        351.4     +15.0%
Total output tokens              33976        32337


## Benchmark 3 — All-Layer Activation Capture

Capture residual-stream activations from all 80 layers. Worst-case capture overhead.

**Note:** Uses fewer prompts and shorter generation than other benchmarks because
capturing all 80 layers stores ~195 MB of activations per request on CPU
(80 × seq_len × 8192 × 2 bytes). 50 requests × 128 tokens ≈ 10 GB peak CPU RAM.

In [8]:
ALL_LAYERS_PROMPTS = NUM_PROMPTS
ALL_LAYERS_MAX_TOKENS = MAX_TOKENS

all_layers_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=ALL_LAYERS_MAX_TOKENS,
    extra_args={"output_residual_stream": True},
)
all_layers = await run_benchmark(
    engine, prompts[:ALL_LAYERS_PROMPTS], lambda i: all_layers_params, "all-layers"
)
compare(baseline, all_layers)

all-layers: 100%|██████████| 50/50 [05:42<00:00,  6.85s/it]


Metric                        Baseline         Lens   Overhead
-------------------------------------------------------------
Wall time (s)                    82.15       342.59    +317.0%
Requests/s                         0.6          0.1
Tokens/s                         413.6         93.9     +77.3%
Total output tokens              33976        32154


## Benchmark 4 — Random Layer Alternation

Each request captures activations from a different random layer (index 0–10).
Simulates a mixed workload where different requests probe different layers.

In [9]:
rng = random.Random(SEED)
random_layers = [rng.randint(0, 10) for _ in range(NUM_PROMPTS)]


def make_random_layer_params(idx: int) -> SamplingParams:
    return SamplingParams(
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        extra_args={"output_residual_stream": [random_layers[idx]]},
    )


random_layer = await run_benchmark(
    engine, prompts, make_random_layer_params, "random-layer"
)
compare(baseline, random_layer)

random-layer: 100%|██████████| 50/50 [01:31<00:00,  1.84s/it]


Metric                        Baseline         Lens   Overhead
-------------------------------------------------------------
Wall time (s)                    82.15        91.78     +11.7%
Requests/s                         0.6          0.5
Tokens/s                         413.6        366.1     +11.5%
Total output tokens              33976        33599


## Benchmark 5 — Steering Vector

Apply a random steering vector at layer 5 with `norm_match=True` and `scale=0.1`.

In [10]:
from vllm_lens._helpers.types import SteeringVector

torch.manual_seed(SEED)
steering_config = [
    SteeringVector(
        activations=torch.randn(1, HIDDEN_DIM),
        layer_indices=[5],
        scale=0.1,
        norm_match=True,
    )
]

steering_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    extra_args={"apply_steering_vectors": steering_config},
)
steering = await run_benchmark(engine, prompts, lambda i: steering_params, "steering")
compare(baseline, steering)

steering: 100%|██████████| 50/50 [01:36<00:00,  1.94s/it]


Metric                        Baseline         Lens   Overhead
-------------------------------------------------------------
Wall time (s)                    82.15        96.92     +18.0%
Requests/s                         0.6          0.5
Tokens/s                         413.6        374.0      +9.6%
Total output tokens              33976        36251


## Benchmark 6 — LoRA Alternation

Alternate a LoRA adapter on/off per request (even index = no LoRA, odd index = LoRA).
Measures the overhead of mixed LoRA/base-model serving.

In [11]:
lora_request = LoRARequest("oracle", 1, LORA_PATH)


def make_lora_alternating(idx: int) -> LoRARequest | None:
    return lora_request if idx % 2 else None


lora_params = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
lora_alt = await run_benchmark(
    engine, prompts, lambda i: lora_params, "lora-alt", make_lora=make_lora_alternating
)
compare(baseline, lora_alt)

lora-alt:   0%|          | 0/50 [00:00<?, ?it/s]

WARNING 03-14 10:44:43 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


lora-alt: 100%|██████████| 50/50 [02:46<00:00,  3.33s/it]


Metric                        Baseline         Lens   Overhead
-------------------------------------------------------------
Wall time (s)                    82.15       166.26    +102.4%
Requests/s                         0.6          0.3
Tokens/s                         413.6        203.6     +50.8%
Total output tokens              33976        33847


## Summary

In [12]:
results = [baseline, one_layer, all_layers, random_layer, steering, lora_alt]

print(
    f"{'Benchmark':<20} {'Time (s)':>10} {'Tok/s':>10} {'Req/s':>10} {'Overhead':>10}"
)
print("=" * 62)
for r in results:
    if r is baseline:
        overhead = "---"
    else:
        pct = (r.wall_time - baseline.wall_time) / baseline.wall_time * 100
        overhead = f"{pct:+.1f}%"
    print(
        f"{r.name:<20} {r.wall_time:>10.2f} {r.tokens_per_second:>10.1f}"
        f" {r.requests_per_second:>10.1f} {overhead:>10}"
    )

Benchmark              Time (s)      Tok/s      Req/s   Overhead
baseline                  82.15      413.6        0.6        ---
one-layer                 92.02      351.4        0.5     +12.0%
all-layers               342.59       93.9        0.1    +317.0%
random-layer              91.78      366.1        0.5     +11.7%
steering                  96.92      374.0        0.5     +18.0%
lora-alt                 166.26      203.6        0.3    +102.4%
